# MIMIC-IV EDA — SepsisAlert
Quick exploratory analysis of the key tables.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

ICU  = 'physionet.org/files/mimiciv/3.1/icu'
HOSP = 'physionet.org/files/mimiciv/3.1/hosp'

con = duckdb.connect()

## 1. ICU Stays overview

In [ ]:
icu_stays = con.execute(f"""
    SELECT first_careunit, COUNT(*) as n_stays, AVG(los) as avg_los_days
    FROM read_csv_auto('{ICU}/icustays.csv.gz')
    GROUP BY first_careunit
    ORDER BY n_stays DESC
""").df()
icu_stays

## 2. Sepsis prevalence (ICD-10 A41.x)

In [ ]:
sepsis = con.execute(f"""
    SELECT
        COUNT(DISTINCT hadm_id) as total_admissions,
        COUNT(DISTINCT CASE WHEN icd_code LIKE 'A41%' OR icd_code LIKE 'R652%' THEN hadm_id END) as sepsis_admissions
    FROM read_csv_auto('{HOSP}/diagnoses_icd.csv.gz')
    WHERE icd_version = 10
""").df()
sepsis['prevalence'] = sepsis['sepsis_admissions'] / sepsis['total_admissions']
sepsis

## 3. Sample key lab values (first 1000 rows of lactate)

In [ ]:
# itemid 50813 = Lactate
lactate = con.execute(f"""
    SELECT valuenum
    FROM read_csv_auto('{HOSP}/labevents.csv.gz')
    WHERE itemid = 50813
      AND valuenum IS NOT NULL
      AND valuenum BETWEEN 0 AND 20
    LIMIT 10000
""").df()

lactate['valuenum'].hist(bins=50)
plt.xlabel('Lactate (mmol/L)')
plt.title('Lactate Distribution (sample)')
plt.show()

## 4. Vital signs sample (Heart Rate)

In [ ]:
# itemid 220045 = Heart Rate
hr = con.execute(f"""
    SELECT valuenum
    FROM read_csv_auto('{ICU}/chartevents.csv.gz')
    WHERE itemid = 220045
      AND valuenum IS NOT NULL
      AND valuenum BETWEEN 20 AND 250
    LIMIT 10000
""").df()

hr['valuenum'].hist(bins=50)
plt.xlabel('Heart Rate (bpm)')
plt.title('Heart Rate Distribution (sample)')
plt.show()